# LingBot 实时视频 + 3D map 网页预览（Viser 双栏版）

这个 notebook 会：
1. 使用你自己的真实室内视频或连续图片序列作为输入（`VIDEO_PATH` / `IMAGE_DIR`）；
2. 复用 `external-lib/lingbot-map/demo.py` 里的函数，调用 LingBot 的 `GCTStream` 做流式 3D 重建；
3. 用官方 `PointCloudViewer`（基于 viser）打开一个双栏页面：
   - 左侧 GUI 面板的 "Current Frame" 会自动显示当前这一帧的真实视频/图片；
   - 右侧主视图显示实时更新的 3D 点云 map。

注意：直接在下面的变量里填写 `VIDEO_PATH` 或 `IMAGE_DIR`，不要写死在代码里；如果它们为空，就会自动回退到仓库里的示例素材。

In [1]:
from pathlib import Path
import os
import json
import shutil
import subprocess
import sys
import time
import numpy as np
import cv2
import torch

ROOT = Path('/home/snt/projects/AgenticMemoryNav').resolve()
LINGBOT_ROOT = ROOT / 'external-lib' / 'lingbot-map'
if str(LINGBOT_ROOT) not in sys.path:
    sys.path.insert(0, str(LINGBOT_ROOT))

# 直接在这里填写你自己的路径：
# VIDEO_PATH = '/mnt/data/room_001.mp4'
# IMAGE_DIR = '/home/snt/your_sequence_dir'

VIDEO_PATH = '/home/snt/projects/AgenticMemoryNav/external-lib/lingbot-map/example/courthouse'
IMAGE_DIR = ''

# 允许用户在这里直接写入字符串路径；为空时回退到示例素材
VIDEO_PATH = Path(VIDEO_PATH).expanduser() if VIDEO_PATH.strip() else None
IMAGE_DIR = Path(IMAGE_DIR).expanduser() if IMAGE_DIR.strip() else None

# 常见误填：把图片目录写进了 VIDEO_PATH（目录 .exists() 也是 True，但 cv2 打不开）。
# 这里自动纠正：如果 VIDEO_PATH 其实是个目录，就当作 IMAGE_DIR 使用。
if VIDEO_PATH is not None and VIDEO_PATH.is_dir() and IMAGE_DIR is None:
    IMAGE_DIR = VIDEO_PATH
    VIDEO_PATH = None
if VIDEO_PATH is not None and not VIDEO_PATH.is_file():
    VIDEO_PATH = None

if VIDEO_PATH is None:
    fallback_video = ROOT / 'outputs' / 'lingbot_demo' / 'courthouse_demo.mp4'
    VIDEO_PATH = fallback_video if fallback_video.is_file() else None

if IMAGE_DIR is None:
    fallback_image_dir = ROOT / 'external-lib' / 'lingbot-map' / 'example' / 'loop'
    if not fallback_image_dir.is_dir():
        fallback_image_dir = ROOT / 'external-lib' / 'lingbot-map' / 'example' / 'courthouse'
    IMAGE_DIR = fallback_image_dir if fallback_image_dir.is_dir() else None

MODEL_PATH = ROOT / 'external-lib' / 'lingbot-map' / 'models' / 'lingbot-map' / 'lingbot-map.pt'
VENV_PYTHON = ROOT / '.lingbot-venv' / 'bin' / 'python'

assert MODEL_PATH.exists(), f'Model not found: {MODEL_PATH}'
print('ROOT:', ROOT)
print('VIDEO_PATH:', VIDEO_PATH)
print('IMAGE_DIR:', IMAGE_DIR)
print('MODEL_PATH:', MODEL_PATH)
print('VENV_PYTHON:', VENV_PYTHON)

if VIDEO_PATH is not None and VIDEO_PATH.is_file():
    print('Video exists:', VIDEO_PATH, 'size_mb=', round(VIDEO_PATH.stat().st_size / (1024 * 1024), 2))
elif IMAGE_DIR is not None and IMAGE_DIR.is_dir():
    imgs = sorted(IMAGE_DIR.glob('*'))
    imgs = [p for p in imgs if p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}]
    print(f'Image-sequence dir detected: {IMAGE_DIR}, count={len(imgs)}')
    print('First few files:', [p.name for p in imgs[:5]])
else:
    raise FileNotFoundError('No valid video or image sequence found. Please fill VIDEO_PATH or IMAGE_DIR with a real path.')


ROOT: /home/snt/projects/AgenticMemoryNav
VIDEO_PATH: /home/snt/projects/AgenticMemoryNav/outputs/lingbot_demo/courthouse_demo.mp4
IMAGE_DIR: /home/snt/projects/AgenticMemoryNav/external-lib/lingbot-map/example/courthouse
MODEL_PATH: /home/snt/projects/AgenticMemoryNav/external-lib/lingbot-map/models/lingbot-map/lingbot-map.pt
VENV_PYTHON: /home/snt/projects/AgenticMemoryNav/.lingbot-venv/bin/python
Video exists: /home/snt/projects/AgenticMemoryNav/outputs/lingbot_demo/courthouse_demo.mp4 size_mb= 4.55


In [2]:
# --- 1) 真实室内视频 / 连续图片准备 ---
# 支持两种输入：
#   1) 一段真实视频: VIDEO_PATH = '/path/to/video.mp4'
#   2) 连续图片序列: IMAGE_DIR = '/path/to/sequence_dir'
#
# 你可以直接修改下面这两行，或者在上一格中填入真实路径。


def pick_media_source(video_path: Path | None, image_dir: Path | None = None):
    # 目录被误填进 video_path 时（.exists() 为 True 但不是文件），当作 image_dir 处理
    if video_path is not None and video_path.is_dir() and image_dir is None:
        image_dir = video_path
        video_path = None
    if video_path is not None and video_path.is_file():
        return 'video', video_path, None
    if image_dir is not None and image_dir.is_dir():
        imgs = sorted(image_dir.glob('*'))
        imgs = [p for p in imgs if p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}]
        if imgs:
            return 'images', None, image_dir
    raise FileNotFoundError(f'No valid video or image sequence found. video={video_path}, image_dir={image_dir}')

media_mode, VIDEO_PATH, IMAGE_DIR = pick_media_source(VIDEO_PATH, IMAGE_DIR)

print('media_mode =', media_mode)
if media_mode == 'video':
    print('Using video:', VIDEO_PATH)
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f'Video frame count: {count}')
    cap.release()
else:
    imgs = sorted(IMAGE_DIR.glob('*'))
    imgs = [p for p in imgs if p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}]
    print('Using image sequence:', IMAGE_DIR)
    print(f'Image count: {len(imgs)}')
    print('First 5:', [p.name for p in imgs[:5]])


media_mode = video
Using video: /home/snt/projects/AgenticMemoryNav/outputs/lingbot_demo/courthouse_demo.mp4
Video frame count: 286


In [3]:
# --- 2) 加载 LingBot 模型（真实 checkpoint） ---
import importlib.util

assert MODEL_PATH.exists(), f'Model missing: {MODEL_PATH}'
assert VENV_PYTHON.exists(), f'Python env missing: {VENV_PYTHON}'

print('Check torch in venv...')
cmd = [
    str(VENV_PYTHON),
    '-c',
    'import torch; import lingbot_map; import viser; print(torch.__version__); print(torch.cuda.is_available()); print(viser.__version__)'
]
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout.strip())
print(res.stderr.strip())
print('returncode=', res.returncode)

if res.returncode != 0:
    raise RuntimeError('LingBot venv is not healthy; install torch + lingbot_map + viser first.')

print('LingBot runtime appears OK.')


Check torch in venv...
2.8.0+cu128
True
1.1.0

returncode= 0
LingBot runtime appears OK.


In [4]:
# --- 3) 在 notebook 内直接跑推理 + viser 双栏可视化（真实视频 + 3D map）---
# 复用 external-lib/lingbot-map/demo.py 里的函数（load_images / load_model / postprocess /
# prepare_for_visualization）和官方的 PointCloudViewer，而不是另开子进程：
#   1) load_images(...)          按 media_mode 加载 VIDEO_PATH 或 IMAGE_DIR
#   2) load_model(...)           加载 GCTStream + checkpoint
#   3) model.inference_streaming(...)  流式 3D 重建
#   4) PointCloudViewer(...)     用 viser 渲染双栏页面：
#        - 左侧 GUI 面板 "Current Frame"：自动显示当前这一帧的真实视频/图片
#        - 右侧主视图：3D 点云 map，随时间步一起播放
# 这本身就是 viser 原生的"真实视频 + 3D map"双栏效果，会自动使用当前 VIDEO_PATH / IMAGE_DIR。

import argparse
import threading
import time

import torch

import demo as lingbot_demo
from lingbot_map.vis import PointCloudViewer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if media_mode == 'video':
    load_kwargs = dict(video_path=str(VIDEO_PATH), fps=10)
    source_desc = f'video: {VIDEO_PATH}'
else:
    load_kwargs = dict(image_folder=str(IMAGE_DIR))
    source_desc = f'image_folder: {IMAGE_DIR}'

print('Source:', source_desc)
print('Loading frames...')
images, paths, resolved_image_folder = lingbot_demo.load_images(**load_kwargs)

args = argparse.Namespace(
    mode='streaming',
    image_size=518,
    patch_size=14,
    enable_3d_rope=True,
    max_frame_num=1024,
    kv_cache_sliding_window=64,
    num_scale_frames=8,
    camera_num_iterations=4,
    use_sdpa=True,
    model_path=str(MODEL_PATH),
)

print('Loading model...')
model = lingbot_demo.load_model(args, device)

if torch.cuda.is_available():
    dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
else:
    dtype = torch.float32
if dtype != torch.float32 and getattr(model, 'aggregator', None) is not None:
    model.aggregator = model.aggregator.to(dtype=dtype)

images = images.to(device)
num_frames = images.shape[0]
keyframe_interval = 1 if num_frames <= 320 else (num_frames + 319) // 320
print(f'Input: {num_frames} frames, keyframe_interval={keyframe_interval}')

print(f'Running streaming inference (dtype={dtype})...')
t0 = time.time()
with torch.no_grad(), torch.amp.autocast('cuda', dtype=dtype):
    predictions = model.inference_streaming(
        images,
        num_scale_frames=8,
        keyframe_interval=keyframe_interval,
    )
print(f'Inference done in {time.time() - t0:.1f}s')

predictions, images_cpu = lingbot_demo.postprocess(predictions, images)
pred_dict = lingbot_demo.prepare_for_visualization(predictions, images_cpu)

viewer = PointCloudViewer(
    pred_dict=pred_dict,
    port=8080,
    vis_threshold=1.5,
    downsample_factor=10,
    point_size=0.00001,
    mask_sky=True,
    image_folder=resolved_image_folder,
)

# animate() 内部是一个 while True 播放循环，放到后台线程里跑，
# 这样这个单元格能立刻返回，viser 服务继续在后台提供页面。
viewer_thread = threading.Thread(target=viewer.animate, daemon=True)
viewer_thread.start()

print('Viser 双栏页面已启动: http://localhost:8080')
print('左侧 "Current Frame" 面板 = 真实视频/图片当前帧，右侧主视图 = 3D map。')


/home/snt/projects/AgenticMemoryNav/.lingbot-venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Source: video: /home/snt/projects/AgenticMemoryNav/outputs/lingbot_demo/courthouse_demo.mp4
Loading frames...


Extracting frames: 100%|██████████| 286/286 [00:00<00:00, 707.39frame/s]


Extracted 286 frames from video (286 total, interval=1)
Loading 286 images...


Loading images: 100%|██████████| 286/286 [00:00<00:00, 557.81it/s]


...

Sky segmentation applied successfully
Viser 双栏页面已启动: http://localhost:8080
左侧 "Current Frame" 面板 = 真实视频/图片当前帧，右侧主视图 = 3D map。


(viser) Connection opened (0, 1 total), 3788 persistent messages

(viser) Connection opened (1, 2 total), 3788 persistent messages

(viser) Connection closed (1, 1 total)

(viser) Connection closed (0, 0 total)